In [ ]:
# ---- Configuration: edit config/config.yaml (see README) ----
# DrugA cohort, serum + CSF: same analysis as 02a (version 2)
# Input files are read from, and outputs written to, <druga_workdir> (see README for the file list).
import os, sys
from pathlib import Path
_root = Path(os.environ.get("PROJECT_ROOT") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "config.py").exists()))
os.environ["PROJECT_ROOT"] = str(_root)
sys.path.insert(0, str(_root))
from utils.config import cfg, cfg_path
os.chdir(cfg_path("druga_workdir", create=True))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean, squareform
from scipy.cluster import hierarchy
import textwrap
from scipy.spatial.distance import euclidean
from scipy.cluster.hierarchy import linkage, dendrogram  # Add this import statement
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
Data_serum = pd.read_csv("Data_serum_filtered.csv", index_col=0)
Data_serum.head(7000000)

In [ ]:
Data_CSF = pd.read_csv("Data_CSF_filtered.csv", index_col=0)
Data_CSF.head(7000000)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Assuming 'Symbol' column contains gene symbols in your DataFrames

# Extracting all columns from each set of significant genes
gene_data1 = Data_serum.copy()
gene_data2 = Data_CSF.copy()


In [ ]:
# Calculate the total number of unique genes
total_genes_sigs = len(set.union(set(gene_data1.index), set(gene_data2.index)))

# Create a Venn diagram with equal-sized circles
venn_labels_sigs = {'10': len(set(gene_data1.index) - set(gene_data2.index)),
                    '01': len(set(gene_data2.index) - set(gene_data1.index)),
                    '11': len(set(gene_data1.index) & set(gene_data2.index))}

venn_labels_sigs = {key: f"{value}" for key, value in venn_labels_sigs.items()}



# Create a Venn diagram
venn_sigs = venn2([set(gene_data1.index), set(gene_data2.index)],
                  set_labels=('Serum', 'CSF'),
                  normalize_to=total_genes_sigs)

# Set labels for each region in the Venn diagram
for key, value in venn_labels_sigs.items():
    text = venn_sigs.get_label_by_id(key)
    text.set_text(value)
    text.set_fontweight('bold')  # set text to bold

# Set all text in the plot to bold
for text in venn_sigs.set_labels + venn_sigs.subset_labels:
    text.set_fontweight('bold')

# Display the plot
plt.title("Venn Diagram", y=1.08, fontweight='bold')

plt.savefig('Venn Diagram.png')

plt.show()

In [ ]:
import pandas as pd

# Load your dataframes
gene_data1 = Data_serum.copy()
gene_data2 = Data_CSF.copy()

# Find common genes using index column
Serum_CSF_common_proteins = pd.merge(gene_data1, gene_data2, left_index=True, right_index=True, how="inner")

# Find genes only in dataframe 1
Serum_only = gene_data1[~gene_data1.index.isin(Serum_CSF_common_proteins.index)]

# Find genes only in dataframe 2
CSF_only = gene_data2[~gene_data2.index.isin(Serum_CSF_common_proteins.index)]

# Save them as CSV files
Serum_CSF_common_proteins.to_csv("Serum_CSF_common_proteins.csv")
Serum_only.to_csv("Serum_only.csv")
CSF_only.to_csv("CSF_only.csv")

print(Serum_CSF_common_proteins)
print(Serum_only)
print(CSF_only)


In [ ]:
Serum_CSF_common_proteins

In [ ]:
renamed_columns = {
    'BL1_x': 'BL1_Serum',
    'BL2_x': 'BL2_Serum',
    'BL3_x': 'BL3_Serum',
    'BL4_x': 'BL4_Serum',
    'BL5_x': 'BL5_Serum',
    'BL6_x': 'BL6_Serum',
    'BL7_x': 'BL7_Serum',
    'BL8_x': 'BL8_Serum',
    'BL9_x': 'BL9_Serum',
    'DrugA1_x': 'DrugA1_Serum',
    'DrugA2_x': 'DrugA2_Serum',
    'DrugA3_x': 'DrugA3_Serum',
    'DrugA4_x': 'DrugA4_Serum',
    'DrugA5_x': 'DrugA5_Serum',
    'DrugA6_x': 'DrugA6_Serum',
    'DrugA7_x': 'DrugA7_Serum',
    'DrugA8_x': 'DrugA8_Serum',
    'DrugA9_x': 'DrugA9_Serum',
    'Ctrl1_x': 'HC1_Serum',
    'Ctrl2_x': 'HC2_Serum',
    'Ctrl3_x': 'HC3_Serum',
    'Ctrl4_x': 'HC4_Serum',
    'Ctrl5_x': 'HC5_Serum',
    'Ctrl6_x': 'HC6_Serum',
    'Ctrl7_x': 'HC7_Serum',
    'Ctrl8_x': 'HC8_Serum',
    'Ctrl9_x': 'HC9_Serum',
    'BL1_y': 'BL1_CSF',
    'BL2_y': 'BL2_CSF',
    'BL3_y': 'BL3_CSF',
    'BL4_y': 'BL4_CSF',
    'BL5_y': 'BL5_CSF',
    'BL6_y': 'BL6_CSF',
    'BL7_y': 'BL7_CSF',
    'BL8_y': 'BL8_CSF',
    'BL9_y': 'BL9_CSF',
    'DrugA1_y': 'DrugA1_CSF',
    'DrugA2_y': 'DrugA2_CSF',
    'DrugA3_y': 'DrugA3_CSF',
    'DrugA4_y': 'DrugA4_CSF',
    'DrugA5_y': 'DrugA5_CSF',
    'DrugA6_y': 'DrugA6_CSF',
    'DrugA7_y': 'DrugA7_CSF',
    'DrugA8_y': 'DrugA8_CSF',
    'DrugA9_y': 'DrugA9_CSF',
    'Ctrl1_y': 'HC1_CSF',
    'Ctrl2_y': 'HC2_CSF',
    'Ctrl3_y': 'HC3_CSF',
    'Ctrl4_y': 'HC4_CSF',
    'Ctrl5_y': 'HC5_CSF',
    'Ctrl6_y': 'HC6_CSF',
    'Ctrl7_y': 'HC7_CSF',
    'Ctrl8_y': 'HC8_CSF',
    'Ctrl9_y': 'HC9_CSF',
}

# Rename the columns
Serum_CSF_common_proteins = Serum_CSF_common_proteins.rename(columns=renamed_columns)
Serum_CSF_common_proteins

In [ ]:
# Calculate z-scores for the dataframe
Serum_CSF_common_proteins_zscore = ((Serum_CSF_common_proteins - Serum_CSF_common_proteins.median()) / Serum_CSF_common_proteins.median())*100

from matplotlib.colors import LinearSegmentedColormap

# Define a custom colormap from blue to white to red
cmap_colors = [(0, 'blue'), (0.5, 'white'), (1, 'red')]
custom_cmap = LinearSegmentedColormap.from_list("custom_colormap", cmap_colors)

# Create a clustermap based on z-scores
sns.clustermap(Serum_CSF_common_proteins_zscore, cmap=custom_cmap, figsize=(20,50))
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define a custom colormap from blue to white to red
cmap_colors = [(0, 'blue'), (0.5, 'white'), (1, 'red')]
custom_cmap = LinearSegmentedColormap.from_list("custom_colormap", cmap_colors)

# Assuming Serum_CSF_common_proteins is your data
Serum_CSF_common_proteins_clustermap = sns.clustermap(Serum_CSF_common_proteins, annot=False, fmt=".4f", cmap=custom_cmap, xticklabels=True, yticklabels=True, figsize=(12, 18))
plt.savefig('Serum_CSF_clustermap.png')

plt.show()


In [ ]:
selected_columns1 = ['HC1_Serum', 'HC2_Serum', 'HC3_Serum', 'HC4_Serum', 'HC5_Serum', 'HC6_Serum', 'HC7_Serum', 'HC8_Serum', 'HC9_Serum', 'HC1_CSF', 'HC2_CSF', 'HC3_CSF', 'HC4_CSF', 'HC5_CSF', 'HC6_CSF', 'HC7_CSF', 'HC8_CSF', 'HC9_CSF']
  # Replace with your column names
selected_data1 = Serum_CSF_common_proteins[selected_columns1]

# Step 3: Calculate correlation matrix
correlation_matrix1 = selected_data1.corr()
correlation_matrix1

In [ ]:
# Plot the clustermap with consistent x and y tick labels
Ctrl_clustermap = sns.clustermap(correlation_matrix1, annot=False, fmt=".4f", cmap="Greens", xticklabels=True, yticklabels=True)
plt.savefig('Ctrl_clustermap.png')

In [ ]:
selected_columns2 =['BL1_Serum', 'BL2_Serum', 'BL3_Serum', 'BL4_Serum', 'BL5_Serum', 'BL6_Serum', 'BL7_Serum', 'BL8_Serum', 'BL9_Serum', 'BL1_CSF', 'BL2_CSF', 'BL3_CSF', 'BL4_CSF', 'BL5_CSF', 'BL6_CSF', 'BL7_CSF', 'BL8_CSF', 'BL9_CSF']

  # Replace with your column names
selected_data2 = Serum_CSF_common_proteins[selected_columns2]

# Step 3: Calculate correlation matrix
correlation_matrix2 = selected_data2.corr()
correlation_matrix2


In [ ]:
# Plot the clustermap with consistent x and y tick labels
BL_clustermap = sns.clustermap(correlation_matrix2, annot=False, fmt=".4f", cmap="Greens", xticklabels=True, yticklabels=True)
plt.savefig('BL_clustermap.png')

In [ ]:
selected_columns3 =['DrugA1_Serum', 'DrugA2_Serum', 'DrugA3_Serum', 'DrugA4_Serum', 'DrugA5_Serum', 'DrugA6_Serum', 'DrugA7_Serum', 'DrugA8_Serum', 'DrugA9_Serum', 'DrugA1_CSF', 'DrugA2_CSF', 'DrugA3_CSF', 'DrugA4_CSF', 'DrugA5_CSF', 'DrugA6_CSF', 'DrugA7_CSF', 'DrugA8_CSF', 'DrugA9_CSF']
  # Replace with your column names
selected_data3 = Serum_CSF_common_proteins[selected_columns3]

# Step 3: Calculate correlation matrix
correlation_matrix3 = selected_data3.corr()
correlation_matrix3


In [ ]:
# Plot the clustermap with consistent x and y tick labels
DrugA_clustermap = sns.clustermap(correlation_matrix3, annot=False, fmt=".4f", cmap="Greens", xticklabels=True, yticklabels=True)
plt.savefig('DrugA_clustermap.png')

In [ ]:
Serum_CSF_common_proteins

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Serum_CSF_common_proteins)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1_Serum': 'MediumVioletRed',
    'BL2_Serum': 'MediumVioletRed',
    'BL3_Serum': 'MediumVioletRed',
    'BL4_Serum': 'MediumVioletRed',
    'BL5_Serum': 'MediumVioletRed',
    'BL6_Serum': 'MediumVioletRed',
    'BL7_Serum': 'MediumVioletRed',
    'BL8_Serum': 'MediumVioletRed',
    'BL9_Serum': 'MediumVioletRed',
    'DrugA1_Serum': 'lightpink',
    'DrugA2_Serum': 'lightpink',
    'DrugA3_Serum': 'lightpink',
    'DrugA4_Serum': 'lightpink',
    'DrugA5_Serum': 'lightpink',
    'DrugA6_Serum': 'lightpink',
    'DrugA7_Serum': 'lightpink',
    'DrugA8_Serum': 'lightpink',
    'DrugA9_Serum': 'lightpink',
    'HC1_Serum': 'DimGray',
    'HC2_Serum': 'DimGray',
    'HC3_Serum': 'DimGray',
    'HC4_Serum': 'DimGray',
    'HC5_Serum': 'DimGray',
    'HC6_Serum': 'DimGray',
    'HC7_Serum': 'DimGray',
    'HC8_Serum': 'DimGray',
    'HC9_Serum': 'DimGray',
    'BL1_CSF': 'MediumVioletRed',
    'BL2_CSF': 'MediumVioletRed',
    'BL3_CSF': 'MediumVioletRed',
    'BL4_CSF': 'MediumVioletRed',
    'BL5_CSF': 'MediumVioletRed',
    'BL6_CSF': 'MediumVioletRed',
    'BL7_CSF': 'MediumVioletRed',
    'BL8_CSF': 'MediumVioletRed',
    'BL9_CSF': 'MediumVioletRed',
    'DrugA1_CSF': 'lightpink',
    'DrugA2_CSF': 'lightpink',
    'DrugA3_CSF': 'lightpink',
    'DrugA4_CSF': 'lightpink',
    'DrugA5_CSF': 'lightpink',
    'DrugA6_CSF': 'lightpink',
    'DrugA7_CSF': 'lightpink',
    'DrugA8_CSF': 'lightpink',
    'DrugA9_CSF': 'lightpink',
    'HC1_CSF': 'DimGray',
    'HC2_CSF': 'DimGray',
    'HC3_CSF': 'DimGray',
    'HC4_CSF': 'DimGray',
    'HC5_CSF': 'DimGray',
    'HC6_CSF': 'DimGray',
    'HC7_CSF': 'DimGray',
    'HC8_CSF': 'DimGray',
    'HC9_CSF': 'DimGray',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1_Serum': 'Baseline - Serum',
    'DrugA1_Serum': 'DrugA - Serum',
    'HC1_Serum': 'Control- Serum',
    'BL1_CSF': 'Baseline - CSF',
    'DrugA1_CSF': 'DrugA - CSF',
    'HC1_CSF': 'Control - CSF',
}

# Custom labels for specific data points
custom_labels = {
    'BL1_Serum': 'Baseline - Serum',
    'DrugA1_Serum': 'DrugA - Serum',
    'HC1_Serum': 'Control- Serum',
    'BL1_CSF': 'Baseline - CSF',
    'DrugA1_CSF': 'DrugA - CSF',
    'HC1_CSF': 'Control - CSF',
    # Add more custom labels as needed
}

# List of markers for each group of selected columns
markers = ['^', 's', 'o']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Serum_CSF_common_proteins.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Serum_CSF_common_proteins.columns):
    color = column_colors.get(col_name, 'gray')
    #marker = markers[i // 9]  # assign a marker based on the group
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5)
   #plt.text(pca_2_result[i, 0]-0.1, pca_2_result[i, 1]+0.1, custom_labels.get(col_name, col_name), fontsize=12, color='black', ha='center', va='center')


# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='center', va='top')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Create custom legend handles
legend_handles = []
for marker, color, label in zip(markers, ['MediumVioletRed', 'lightpink', 'DimGray'], ['Baseline', 'DrugA', 'Control']):
    legend_handles.append(plt.Line2D([0], [0], marker=marker, color='w', markerfacecolor=color, markersize=10, label=label))

# Display the legend with custom handles outside the plot
plt.legend(handles=legend_handles, title='Legend', bbox_to_anchor=(1.05, 1), loc='upper left')


plt.tight_layout()
plt.savefig('Serum_CSF_PCA_PC1_PC2.png')

# Show the plot
plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# List of markers for each group of selected columns
markers = ['^', 's', 'o']  # Circles for control, squares for baseline, triangles for drugA

# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Serum_CSF_common_proteins.columns)

# Set up the plotting environment
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Serum_CSF_common_proteins.columns):
    marker = markers[i % len(markers)]  # Cycle through markers
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5)

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    ax.text(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1, PC2, and PC3 values
ax.set_xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)', fontsize=10)
ax.set_ylabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)', fontsize=10)
ax.set_zlabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)', fontsize=10)

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('3D Principal Component Analysis', fontweight='bold')

# Add grids
ax.grid(True)

# Create custom legend handles
legend_handles = []
for marker, color, label in zip(markers, ['MediumVioletRed', 'lightpink', 'DimGray'], ['Control', 'Baseline', 'DrugA']):
    legend_handles.append(plt.Line2D([0], [0], marker=marker, color='w', markerfacecolor=color, markersize=10, label=label))

# Display the legend with custom handles outside the plot
plt.legend(handles=legend_handles, title='Legend', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('Serum_CSF_3D_PCA.png')

# Show the plot
plt.show()


In [ ]:
# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Extract PC loadings
serum_csf_all_pc_loadings = pca_3.components_

# Create a DataFrame with PC loadings for all genes
serum_csf_all_pc_loadings = pd.DataFrame(serum_csf_all_pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Serum_CSF_common_proteins.index)
serum_csf_all_pc_loadings

In [ ]:
# Convert the Series serum_csf_all_pc1_sorted to a DataFrame
serum_csf_all_pc1_sorted_df = pd.DataFrame(serum_csf_all_pc_loadings, columns=['PC1'])
serum_csf_all_pc1_sorted_df

# Sort PC loadings by PC1 in descending order
serum_csf_all_pc1_loading_df_sorted = serum_csf_all_pc1_sorted_df.sort_values(by='PC1', ascending=False)
serum_csf_all_pc1_loading_df_sorted.to_csv('serum_csf_all_pc1_loading_df_sorted.csv')
serum_csf_all_pc1_loading_df_sorted


In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=serum_csf_all_pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=serum_csf_all_pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(serum_csf_all_pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(serum_csf_all_pc1_loading_df_sorted.index, serum_csf_all_pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(serum_csf_all_pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('Serum CSF PC1 loadings.png')
plt.show()


In [ ]:
# Convert the Series serum_csf_all_pc2_sorted to a DataFrame
serum_csf_all_pc2_sorted_df = pd.DataFrame(serum_csf_all_pc_loadings, columns=['PC2'])
serum_csf_all_pc2_sorted_df

# Sort PC loadings by PC2 in descending order
serum_csf_all_pc2_loading_df_sorted = serum_csf_all_pc2_sorted_df.sort_values(by='PC2', ascending=False)
serum_csf_all_pc2_loading_df_sorted.to_csv('serum_csf_all_pc2_loading_df_sorted.csv')
serum_csf_all_pc2_loading_df_sorted


In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=serum_csf_all_pc2_loading_df_sorted['PC2'].min(),
                                   vcenter=0,
                                   vmax=serum_csf_all_pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(serum_csf_all_pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(serum_csf_all_pc2_loading_df_sorted.index, serum_csf_all_pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(serum_csf_all_pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('Serum CSF PC2 loadings.png')

plt.show()


In [ ]:
# Convert the Series serum_csf_all_pc3_sorted to a DataFrame
serum_csf_all_pc3_sorted_df = pd.DataFrame(serum_csf_all_pc_loadings, columns=['PC3'])
serum_csf_all_pc3_sorted_df

# Sort PC loadings by PC3 in descending order
serum_csf_all_pc3_loading_df_sorted = serum_csf_all_pc3_sorted_df.sort_values(by='PC3', ascending=False)
serum_csf_all_pc3_loading_df_sorted.to_csv('serum_csf_all_pc3_loading_df_sorted.csv')
serum_csf_all_pc3_loading_df_sorted


In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=serum_csf_all_pc3_loading_df_sorted['PC3'].min(),
                                   vcenter=0,
                                   vmax=serum_csf_all_pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(serum_csf_all_pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(serum_csf_all_pc3_loading_df_sorted.index, serum_csf_all_pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(serum_csf_all_pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('Serum CSF PC3 loadings.png')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Serum_CSF_common_proteins)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(Serum_CSF_common_proteins.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(15, 8))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(Serum_CSF_common_proteins.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(Serum_CSF_common_proteins.columns) + 1))
ax.grid(True)

plt.tight_layout()
plt.savefig('Serum_CSF_PC_Bar_grah.png')

plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_4 = PCA(n_components=4)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_4.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_4_result = pca_4.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_4 = pd.DataFrame(pca_4_result, columns=['PC1', 'PC2', 'PC3', 'PC4'], index=Serum_CSF_common_proteins.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Serum_CSF_common_proteins.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_4_result[i, 2], y=pca_4_result[i, 3], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_4_result[i, 2], y=pca_4_result[i, 3], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    x, y = pca_4_result[i, 2], pca_4_result[i, 3]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC3 and PC4 values
plt.xlabel(f'Principal Component 3 ({pca_4.explained_variance_ratio_[2]*100:.2f}%)')
plt.ylabel(f'Principal Component 4 ({pca_4.explained_variance_ratio_[3]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC3: {pca_4.explained_variance_ratio_[2]*100:.2f}%")
print(f"Explained Variance Ratio PC4: {pca_4.explained_variance_ratio_[3]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC3 vs PC4)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_CSF_PCA_PC3_PC4.png')


# Show the plot
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_5 = PCA(n_components=5)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_5.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_5_result = pca_5.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_5 = pd.DataFrame(pca_5_result, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5'], index=Serum_CSF_common_proteins.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Serum_CSF_common_proteins.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_5_result[i, 3], y=pca_5_result[i, 4], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_5_result[i, 3], y=pca_5_result[i, 4], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    x, y = pca_5_result[i, 3], pca_5_result[i, 4]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC4 and PC5 values
plt.xlabel(f'Principal Component 4 ({pca_5.explained_variance_ratio_[3]*100:.2f}%)')
plt.ylabel(f'Principal Component 5 ({pca_5.explained_variance_ratio_[4]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC4: {pca_5.explained_variance_ratio_[3]*100:.2f}%")
print(f"Explained Variance Ratio PC5: {pca_5.explained_variance_ratio_[4]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC4 vs PC5)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_CSF_PCA_PC4_PC5.png')

# Show the plot
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Serum_CSF_common_proteins.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Serum_CSF_common_proteins.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    x, y = pca_3_result[i, 1], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC2 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_CSF_PCA_PC2_PC3.png')


# Show the plot
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Serum_CSF_common_proteins.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Serum_CSF_common_proteins.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Serum_CSF_common_proteins.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_CSF_PCA_PC1_PC3.png')

# Show the plot
plt.show()


In [ ]:
Data_serum

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Custom labels for columns
custom_labels = {
    'BL1': '1',
    'BL2': '2',
    'BL3': '3',
    'BL4': '4',
    'BL5': '5',
    'BL6': '6',
    'BL7': '7',
    'BL8': '8',
    'BL9': '9',
    'DrugA1': '1',
    'DrugA2': '2',
    'DrugA3': '3',
    'DrugA4': '4',
    'DrugA5': '5',
    'DrugA6': '6',
    'DrugA7': '7',
    'DrugA8': '8',
    'DrugA9': '9',
    'Ctrl1': '1',
    'Ctrl2': '2',
    'Ctrl3': '3',
    'Ctrl4': '4',
    'Ctrl5': '5',
    'Ctrl6': '6',
    'Ctrl7': '7',
    'Ctrl8': '8',
    'Ctrl9': '9',
}

# List of markers for each group of selected columns
markers = ['^', 's', 'o']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    marker = markers[i // 9]  # assign a marker based on the group
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5,label=custom_labels.get(col_name, col_name) if i < 9 else None)
    plt.text(pca_2_result[i, 0]-0.1, pca_2_result[i, 1]+0.1, custom_labels.get(col_name, col_name), fontsize=12, weight='bold', color='black', ha='center', va='center')

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Create custom legend handles
legend_handles = []
for marker, color, label in zip(markers, ['MediumVioletRed', 'lightpink', 'DimGray'], ['Baseline', 'DrugA', 'Control']):
    legend_handles.append(plt.Line2D([0], [0], marker=marker, color='w', markerfacecolor=color, markersize=10, label=label))

# Display the legend with custom handles outside the plot
plt.legend(handles=legend_handles, title='Legend', bbox_to_anchor=(1.05, 1), loc='upper left')


plt.tight_layout()
plt.savefig('Serum_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.columns)

# Set up the plotting environment
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum.columns.get_loc(col_name)
    ax.text(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1, PC2, and PC3 values
ax.set_xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)', fontsize=10)
ax.set_ylabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)', fontsize=10)
ax.set_zlabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)', fontsize=10)

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('3D Principal Component Analysis', fontweight='bold')

# Add grids
ax.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.15, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_3D_PCA.png')


# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_serum.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'white',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_serum.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=0, color=centroid_colors.get(selected_columns[0], 'white'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_centroids_wo_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Custom labels for centroids
centroid_custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=0, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=0, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_serum.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=0)
        plt.gca().add_patch(ellipse)

# Create handles and labels for centroids
centroid_handles = []
centroid_labels = []

for selected_columns in selected_columns_list:
    centroid_handles.append(plt.scatter([], [], marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1))
    centroid_labels.append(centroid_custom_labels[selected_columns[0]])


# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Add centroids to legend with custom labels
plt.legend(handles=centroid_handles, labels=centroid_labels, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_centroids_only_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(Data_serum.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(Data_serum.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(Data_serum.columns) + 1))
ax.grid(True)

plt.tight_layout()
plt.savefig('Serum_PC_bar_graph.png')
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_PCA_PC1_PC3.png')

# Show the plot
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_4 = PCA(n_components=4)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_4.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_4_result = pca_4.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_4 = pd.DataFrame(pca_4_result, columns=['PC1', 'PC2', 'PC3', 'PC4'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_4_result[i, 2], y=pca_4_result[i, 3], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_4_result[i, 2], y=pca_4_result[i, 3], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum.columns.get_loc(col_name)
    x, y = pca_4_result[i, 2], pca_4_result[i, 3]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC3 and PC4 values
plt.xlabel(f'Principal Component 3 ({pca_4.explained_variance_ratio_[2]*100:.2f}%)')
plt.ylabel(f'Principal Component 4 ({pca_4.explained_variance_ratio_[3]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC3: {pca_4.explained_variance_ratio_[2]*100:.2f}%")
print(f"Explained Variance Ratio PC4: {pca_4.explained_variance_ratio_[3]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC3 vs PC4)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_PCA_PC3_PC4.png')

# Show the plot
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum.columns.get_loc(col_name)
    x, y = pca_3_result[i, 1], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC2 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC2', 'PC3']]  # Updated to PC2 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC2'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC2', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('Serum_centroids_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC2', 'PC3']]  # Updated to PC2 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC2'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=0, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC2', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('Serum_centroids_wo_centroids_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# Custom labels for columns
custom_labels = {
    'BL1': '1',
    'BL2': '2',
    'BL3': '3',
    'BL4': '4',
    'BL5': '5',
    'BL6': '6',
    'BL7': '7',
    'BL8': '8',
    'BL9': '9',
    'DrugA1': '1',
    'DrugA2': '2',
    'DrugA3': '3',
    'DrugA4': '4',
    'DrugA5': '5',
    'DrugA6': '6',
    'DrugA7': '7',
    'DrugA8': '8',
    'DrugA9': '9',
    'Ctrl1': 'HC',
    'Ctrl2': 'HC',
    'Ctrl3': 'HC',
    'Ctrl4': 'HC',
    'Ctrl5': 'HC',
    'Ctrl6': 'HC',
    'Ctrl7': 'HC',
    'Ctrl8': 'HC',
    'Ctrl9': 'HC',
}


# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# List of markers for each group of selected columns
markers = ['^', 's', 'o']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    marker = markers[i // 9]  # assign a marker based on the group
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5, label=custom_labels.get(col_name, col_name) if i < 9 else None)
    # Add custom labels for each column
    plt.text(pca_3_result[i, 0]-0.05, pca_3_result[i, 2]+0.05, custom_labels.get(col_name, col_name), fontsize=12, weight='bold', color='black', ha='left', va='bottom')

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC2', 'PC3']]  # Updated to PC2 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC2'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=0, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC2', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Create custom legend handles
legend_handles = []
for marker, color, label in zip(markers, ['MediumVioletRed', 'lightpink', 'DimGray'], ['Baseline', 'DrugA', 'Control']):
    legend_handles.append(plt.Line2D([0], [0], marker=marker, color='w', markerfacecolor=color, markersize=10, label=label))

# Display the legend with custom handles outside the plot
plt.legend(handles=legend_handles, title='Legend', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('Serum_centroids_wo_centroids_shapes_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_serum.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=0, edgecolor='k', linewidth=0.5,)

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC2', 'PC3']]  # Updated to PC2 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC2'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC2', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=0)
        plt.gca().add_patch(ellipse)

# Create handles and labels for centroids
centroid_handles = []
centroid_labels = []

for selected_columns in selected_columns_list:
    centroid_handles.append(plt.scatter([], [], marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1))
    centroid_labels.append(centroid_custom_labels[selected_columns[0]])

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Add centroids to legend with custom labels
plt.legend(handles=centroid_handles, labels=centroid_labels, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')
plt.tight_layout()
plt.savefig('Serum_centroids_only_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Extract PC loadings
serum_all_pc_loadings = pca_3.components_

# Create a DataFrame with PC loadings for all genes
serum_all_pc_loadings = pd.DataFrame(serum_all_pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_serum.index)
serum_all_pc_loadings

In [ ]:
# Convert the Series pc1_sorted to a DataFrame
serum_all_pc1_sorted_df = pd.DataFrame(serum_all_pc_loadings, columns=['PC1'])
serum_all_pc1_sorted_df
# Sort PC loadings by PC1 in descending order
serum_all_pc1_loading_df_sorted = serum_all_pc1_sorted_df.sort_values(by='PC1', ascending=False)
serum_all_pc1_loading_df_sorted.to_csv('serum_all_pc1_loading_df_sorted.csv')
serum_all_pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=serum_all_pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=serum_all_pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(serum_all_pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(serum_all_pc1_loading_df_sorted.index, serum_all_pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(serum_all_pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('Serum PC1 loadings.png')
plt.show()


In [ ]:
# Convert the Series pc2_sorted to a DataFrame
serum_all_pc2_sorted_df = pd.DataFrame(serum_all_pc_loadings, columns=['PC2'])
serum_all_pc2_sorted_df

# Sort PC loadings by PC2 in descending order
serum_all_pc2_loading_df_sorted = serum_all_pc2_sorted_df.sort_values(by='PC2', ascending=False)
serum_all_pc2_loading_df_sorted.to_csv('serum_all_pc2_loading_df_sorted.csv')
serum_all_pc2_loading_df_sorted


In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=serum_all_pc2_loading_df_sorted['PC2'].min(),
                                   vcenter=0,
                                   vmax=serum_all_pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(serum_all_pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(serum_all_pc2_loading_df_sorted.index, serum_all_pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(serum_all_pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('Serum PC2 loadings.png')
plt.show()


In [ ]:
# Convert the Series pc3_sorted to a DataFrame
serum_all_pc3_sorted_df = pd.DataFrame(serum_all_pc_loadings, columns=['PC3'])
serum_all_pc3_sorted_df

# Sort PC loadings by PC3 in descending order
serum_all_pc3_loading_df_sorted = serum_all_pc3_sorted_df.sort_values(by='PC3', ascending=False)
serum_all_pc3_loading_df_sorted.to_csv('serum_all_pc3_loading_df_sorted.csv')
serum_all_pc3_loading_df_sorted


In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=serum_all_pc3_loading_df_sorted['PC3'].min(),
                                   vcenter=0,
                                   vmax=serum_all_pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(serum_all_pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(serum_all_pc3_loading_df_sorted.index, serum_all_pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(serum_all_pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('Serum PC3 loadings.png')
plt.show()


In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'HC1': 'DimGray',
    'HC2': 'DimGray',
    'HC3': 'DimGray',
    'HC4': 'DimGray',
    'HC5': 'DimGray',
    'HC6': 'DimGray',
    'HC7': 'DimGray',
    'HC8': 'DimGray',
    'HC9': 'DimGray',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_CSF.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.columns)

# Set up the plotting environment
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    ax.scatter(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF.columns.get_loc(col_name)
    ax.text(pca_3_result[i, 0], pca_3_result[i, 1], pca_3_result[i, 2], label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1, PC2, and PC3 values
ax.set_xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)', fontsize=10)
ax.set_ylabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)', fontsize=10)
ax.set_zlabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)', fontsize=10)

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('3D Principal Component Analysis', fontweight='bold')

# Add grids
ax.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.15, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_3D_PCA.png')

# Show the plot
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Data_CSF

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_CSF.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_CSF.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=0, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_centroids_wo_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Custom labels for centroids
centroid_custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=0, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=0, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_CSF.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=0)
        plt.gca().add_patch(ellipse)

# Create handles and labels for centroids
centroid_handles = []
centroid_labels = []

for selected_columns in selected_columns_list:
    centroid_handles.append(plt.scatter([], [], marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1))
    centroid_labels.append(centroid_custom_labels[selected_columns[0]])

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Add centroids to legend with custom labels
plt.legend(handles=centroid_handles, labels=centroid_labels, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_centroids_only_PCA_PC1_PC2.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('CSF_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [ ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9'],
                         ['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9']
                        ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=0, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('CSF_centroids_wo_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [ ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9'],
                         ['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9']
                        ]
# Dictionary for centroid colors
centroid_colors = {
    'Ctrl1': 'DimGray',
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    
}

# Custom labels for columns
custom_labels = {
    'BL1': '1',
    'BL2': '2',
    'BL3': '3',
    'BL4': '4',
    'BL5': '5',
    'BL6': '6',
    'BL7': '7',
    'BL8': '8',
    'BL9': '9',
    'DrugA1': '1',
    'DrugA2': '2',
    'DrugA3': '3',
    'DrugA4': '4',
    'DrugA5': '5',
    'DrugA6': '6',
    'DrugA7': '7',
    'DrugA8': '8',
    'DrugA9': '9',
    'Ctrl1': 'HC',
    'Ctrl2': 'HC',
    'Ctrl3': 'HC',
    'Ctrl4': 'HC',
    'Ctrl5': 'HC',
    'Ctrl6': 'HC',
    'Ctrl7': 'HC',
    'Ctrl8': 'HC',
    'Ctrl9': 'HC',
}

# List of outline colors for each cluster
outline_colors = ['DimGray', 'MediumVioletRed', 'lightpink']

# List of markers for each group of selected columns
markers = ['^', 's','o']


# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    marker = markers[i // 9]  # assign a marker based on the group
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5, label=col_name)
    plt.text(pca_3_result[i, 0]-0.05, pca_3_result[i, 2]+0.05, custom_labels.get(col_name, col_name), fontsize=12, weight='bold', color='black', ha='left', va='bottom')


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=0, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)
# Create custom legend handles
legend_handles = []
for marker, color, label in zip(markers, ['MediumVioletRed', 'lightpink','DimGray'], [ 'DrugA','Baseline', 'Control']):
    legend_handles.append(plt.Line2D([0], [0], marker=marker, color='w', markerfacecolor=color, markersize=10, label=label))

# Display the legend with custom handles
plt.legend(handles=legend_handles, title='Legend', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('CSF_centroids_wo_centroids_shapes_PCA_PC1_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink', 'DimGray']

# Custom labels for centroids
centroid_custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
    'Ctrl1': 'Control',
}

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=0, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=0)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
# Add centroids to legend with custom labels
plt.legend(handles=centroid_handles, labels=centroid_labels, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_centroids_only_PCA_PC1_PC3.png')

# Show the plot
plt.show()


In [ ]:
# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Extract PC loadings
csf_all_pc_loadings = pca_3.components_

# Create a DataFrame with PC loadings for all genes
csf_all_pc_loading_df = pd.DataFrame(csf_all_pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.index)
csf_all_pc_loading_df

In [ ]:
# Convert the Series pc1_sorted to a DataFrame
csf_all_pc1_sorted_df = pd.DataFrame(csf_all_pc_loading_df, columns=['PC1'])
csf_all_pc1_sorted_df

In [ ]:
# Sort PC loadings by PC1 in descending order
csf_all_pc1_loading_df_sorted = csf_all_pc1_sorted_df.sort_values(by='PC1', ascending=False)
csf_all_pc1_loading_df_sorted.to_csv('csf_all_pc1_loading_df_sorted.csv')
csf_all_pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=csf_all_pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=csf_all_pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(csf_all_pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(csf_all_pc1_loading_df_sorted.index, csf_all_pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(csf_all_pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('CSF PC1 loadings.png')
plt.show()

In [ ]:
# Convert the Series pc1_sorted to a DataFrame
csf_all_pc3_sorted_df = pd.DataFrame(csf_all_pc_loading_df, columns=['PC3'])
csf_all_pc3_sorted_df


In [ ]:
# Sort PC loadings by PC1 in descending order
csf_all_pc3_loading_df_sorted = csf_all_pc3_sorted_df.sort_values(by='PC3', ascending=False)
csf_all_pc3_loading_df_sorted.to_csv('csf_all_pc3_loading_df_sorted.csv')
csf_all_pc3_loading_df_sorted


In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=csf_all_pc3_loading_df_sorted['PC3'].min(),
                                   vcenter=0,
                                   vmax=csf_all_pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(csf_all_pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(csf_all_pc3_loading_df_sorted.index, csf_all_pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(csf_all_pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('CSF PC3 loadings.png')
plt.show()


In [ ]:
# Convert the Series csf_all_pc2_sorted to a DataFrame
csf_all_pc2_sorted_df = pd.DataFrame(csf_all_pc_loading_df, columns=['PC2'])
csf_all_pc2_sorted_df


In [ ]:

# Sort PC loadings by PC2 in descending order
csf_all_pc2_loading_df_sorted = csf_all_pc2_sorted_df.sort_values(by='PC2', ascending=False)
csf_all_pc2_loading_df_sorted.to_csv('csf_all_pc2_loading_df_sorted.csv')
csf_all_pc2_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=csf_all_pc2_loading_df_sorted['PC2'].min(),
                                   vcenter=0,
                                   vmax=csf_all_pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(csf_all_pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(csf_all_pc2_loading_df_sorted.index, csf_all_pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(csf_all_pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('CSF PC2 loadings.png')
plt.show()


In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF.columns.get_loc(col_name)
    x, y = pca_3_result[i, 1], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC2 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(Data_CSF.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(Data_CSF.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(Data_CSF.columns) + 1))
ax.grid(True)

plt.tight_layout()

plt.savefig('CSF_PC_bar_graph.png')
plt.show()


In [ ]:
Data_serum_filtered_BL_DrugA=pd.read_csv("Data_serum_filtered_BL_DrugA.csv", index_col=0)
Data_serum_filtered_BL_DrugA.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_serum_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_DrugA_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_serum_filtered_BL_DrugA.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_DrugA_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_DrugA_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                        ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_serum_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_DrugA_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Data_serum_sigs_filtered_BL_DrugA=pd.read_csv("Serum sigs BL DrugA.csv", index_col=0)
Data_serum_sigs_filtered_BL_DrugA.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_sigs_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_serum_sigs_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_sigs_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_DrugA_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_sigs_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_sigs_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_serum_sigs_filtered_BL_DrugA.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_DrugA_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum_sigs_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_sigs_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_DrugA_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_sigs_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                        ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum_sigs_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_serum_sigs_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_DrugA_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Data_serum_filtered_BL_Ctrl=pd.read_csv("Data_serum_filtered_BL_Ctrl.csv", index_col=0)
Data_serum_filtered_BL_Ctrl.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_filtered_BL_Ctrl)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_serum_filtered_BL_Ctrl.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_filtered_BL_Ctrl.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum_filtered_BL_Ctrl.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_Ctrl_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_filtered_BL_Ctrl)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_filtered_BL_Ctrl.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_serum_filtered_BL_Ctrl.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_Ctrl_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum_filtered_BL_Ctrl.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_filtered_BL_Ctrl.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum_filtered_BL_Ctrl.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_Ctrl_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_serum_filtered_BL_Ctrl)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum_filtered_BL_Ctrl.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_serum_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_Ctrl_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_serum_filtered_BL_Ctrl.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_serum_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_serum_filtered_BL_Ctrl.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_serum_filtered_BL_Ctrl.columns.get_loc(col_name)
    x, y = pca_3_result[i, 1], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC2 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_BL_Ctrl_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
Data_Serum_sigs_BL_HC=pd.read_csv("Data_Serum_sigs_BL_HC.csv", index_col=0)
Data_Serum_sigs_BL_HC.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_Serum_sigs_BL_HC)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_Serum_sigs_BL_HC.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_Serum_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_Serum_sigs_BL_HC.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_Serum_sigs_BL_HC.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_Ctrl_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_Serum_sigs_BL_HC.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_Serum_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_Serum_sigs_BL_HC.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_Serum_sigs_BL_HC.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_Ctrl_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_Serum_sigs_BL_HC.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_Serum_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_Serum_sigs_BL_HC.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 1], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_Serum_sigs_BL_HC.columns.get_loc(col_name)
    x, y = pca_3_result[i, 1], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC2 and PC3 values
plt.xlabel(f'Principal Component 2 ({pca_3.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC2: {pca_3.explained_variance_ratio_[1]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC2 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_Ctrl_PCA_PC2_PC3.png')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_Serum_sigs_BL_HC)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_Serum_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_Serum_sigs_BL_HC.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_Serum_sigs_BL_HC.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_Ctrl_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_Serum_sigs_BL_HC)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_Serum_sigs_BL_HC.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_Serum_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('Serum_sigs_BL_Ctrl_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Data_CSF_filtered_BL_Ctrl=pd.read_csv("Data_CSF_filtered_BL_Ctrl.csv", index_col=0)
Data_CSF_filtered_BL_Ctrl.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_filtered_BL_Ctrl)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_CSF_filtered_BL_Ctrl.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_filtered_BL_Ctrl.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_filtered_BL_Ctrl.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_Ctrl_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_filtered_BL_Ctrl)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_filtered_BL_Ctrl.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_Serum_sigs_BL_HC.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_Ctrl_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_filtered_BL_Ctrl.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_filtered_BL_Ctrl.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_filtered_BL_Ctrl.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_Ctrl_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_filtered_BL_Ctrl)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_filtered_BL_Ctrl.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF_filtered_BL_Ctrl.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_Ctrl_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Data_CSF_sigs_BL_HC=pd.read_csv("Data_CSF_sigs_BL_HC.csv", index_col=0)
Data_CSF_sigs_BL_HC.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_sigs_BL_HC)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_CSF_sigs_BL_HC.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_sigs_BL_HC.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_sigs_BL_HC.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_Ctrl_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_sigs_BL_HC)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    'Ctrl2': 'DimGray',
    'Ctrl3': 'DimGray',
    'Ctrl4': 'DimGray',
    'Ctrl5': 'DimGray',
    'Ctrl6': 'DimGray',
    'Ctrl7': 'DimGray',
    'Ctrl8': 'DimGray',
    'Ctrl9': 'DimGray',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'Ctrl1': 'Control',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_sigs_BL_HC.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_CSF_sigs_BL_HC.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_Ctrl_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_sigs_BL_HC.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_sigs_BL_HC.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_sigs_BL_HC.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_Ctrl_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_sigs_BL_HC)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['Ctrl1', 'Ctrl2', 'Ctrl3','Ctrl4', 'Ctrl5', 'Ctrl6','Ctrl7', 'Ctrl8', 'Ctrl9']]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'Ctrl1': 'DimGray',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'DimGray']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_sigs_BL_HC.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF_sigs_BL_HC.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'Control']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_Ctrl_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Data_CSF_filtered_BL_DrugA=pd.read_csv("Data_CSF_filtered_BL_DrugA.csv", index_col=0)
Data_CSF_filtered_BL_DrugA.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_CSF_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_DrugA_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_CSF_filtered_BL_DrugA.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_DrugA_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_DrugA_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                        ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('CSF_BL_DrugA_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Data_CSF_sigs_filtered_BL_DrugA=pd.read_csv("CSF sigs BL DrugA.csv", index_col=0)
Data_CSF_sigs_filtered_BL_DrugA.head(7000000)

In [ ]:
# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_sigs_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',
    # Add more columns and colors as needed
}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Custom labels for specific data points
custom_labels = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',

    # Add more custom labels as needed
}

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_2_result, columns=['PC1', 'PC2'], index=Data_CSF_sigs_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_sigs_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_2_result[i, 0], pca_2_result[i, 1]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_2.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC2: {pca_2.explained_variance_ratio_[1]*100:.2f}%")

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_DrugA_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_sigs_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Manually assign colors for each column
column_colors = {
    'BL1': 'MediumVioletRed',
    'BL2': 'MediumVioletRed',
    'BL3': 'MediumVioletRed',
    'BL4': 'MediumVioletRed',
    'BL5': 'MediumVioletRed',
    'BL6': 'MediumVioletRed',
    'BL7': 'MediumVioletRed',
    'BL8': 'MediumVioletRed',
    'BL9': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    'DrugA2': 'lightpink',
    'DrugA3': 'lightpink',
    'DrugA4': 'lightpink',
    'DrugA5': 'lightpink',
    'DrugA6': 'lightpink',
    'DrugA7': 'lightpink',
    'DrugA8': 'lightpink',
    'DrugA9': 'lightpink',

}

# Replace legend names for selected columns
selected_legend_columns = {
    'BL1': 'Baseline',
    'DrugA1': 'DrugA',
}

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                         ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
pca_2 = PCA(n_components=2)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_2.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_2_result = pca_2.transform(data_transposed)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_sigs_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_2_result[i, 0], y=pca_2_result[i, 1], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_2_result[Data_CSF_sigs_filtered_BL_DrugA.columns.isin(selected_columns)]

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_points = selected_data[cluster_indices]

        # Calculate centroid
        centroid_x, centroid_y = np.mean(cluster_points, axis=0)

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC2 values
plt.xlabel(f'Principal Component 1 ({pca_2.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 2 ({pca_2.explained_variance_ratio_[1]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_DrugA_centroids_PCA_PC1_PC2.png')

# Show the plot
plt.show()

In [ ]:
# Create an instance of PCA with the desired number of components
pca_3 = PCA(n_components=3)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df_3 = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_sigs_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for each column (without labels)
for i, col_name in enumerate(Data_CSF_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for col_name in selected_legend_columns:
    i = Data_CSF_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col_name])

# Add custom labels for specific data points
for col_name, label in custom_labels.items():
    i = Data_CSF_sigs_filtered_BL_DrugA.columns.get_loc(col_name)
    x, y = pca_3_result[i, 0], pca_3_result[i, 2]
    plt.text(x, y, label, fontsize=12, ha='left', va='bottom')

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Display explained variance ratio
print(f"Explained Variance Ratio PC1: {pca_3.explained_variance_ratio_[0]*100:.2f}%")
print(f"Explained Variance Ratio PC3: {pca_3.explained_variance_ratio_[2]*100:.2f}%")

# Title
plt.title('Principal Component Analysis (PC1 vs PC3)', fontweight='bold')

# Add grids
plt.grid(True)

# Customized legend box
plt.legend(bbox_to_anchor=(1.05, 0.65), loc='upper left', borderaxespad=0., title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_DrugA_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import chi2
from matplotlib.patches import Ellipse

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(Data_CSF_sigs_filtered_BL_DrugA)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Specify the columns for which you want to create centroids
selected_columns_list = [['BL1', 'BL2', 'BL3','BL4', 'BL5', 'BL6','BL7', 'BL8', 'BL9'],
                         ['DrugA1', 'DrugA2', 'DrugA3','DrugA4', 'DrugA5', 'DrugA6','DrugA7', 'DrugA8', 'DrugA9'],
                        ]

# Dictionary for centroid colors
centroid_colors = {
    'BL1': 'MediumVioletRed',
    'DrugA1': 'lightpink',
    
}

# List of outline colors for each cluster
outline_colors = ['MediumVioletRed', 'lightpink']

# Create an instance of PCA with the desired number of components
num_components = 3  # Change to 3 for PC1 vs PC3
pca_3 = PCA(n_components=num_components)

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_3.fit(data_transposed)

# Transform the data to its principal components without applying the transformation
pca_3_result = pca_3.transform(data_transposed)

# Create a DataFrame with the principal components and corresponding column names
pca_df = pd.DataFrame(pca_3_result, columns=['PC1', 'PC2', 'PC3'], index=Data_CSF_sigs_filtered_BL_DrugA.columns)

# Set up the plotting environment
sns.set(style='white', rc={'figure.figsize': (9, 6)}, font_scale=1.1)

# Scatter plot with manually assigned colors for selected columns (with labels for legend)
for i, col_name in enumerate(Data_CSF_sigs_filtered_BL_DrugA.columns):
    color = column_colors.get(col_name, 'gray')
    plt.scatter(x=pca_3_result[i, 0], y=pca_3_result[i, 2], color=color, s=100, edgecolor='k', linewidth=0.5,)


# Plot centroids for each group of selected columns
for selected_columns, outline_color in zip(selected_columns_list, outline_colors):
    # Extract the data for selected columns
    selected_data = pca_df.loc[selected_columns, ['PC1', 'PC3']]  # Updated to PC1 vs PC3

    # Perform k-means clustering on selected data
    num_clusters = 1  # Set the desired number of clusters
    kmeans_selected = KMeans(n_clusters=num_clusters, random_state=42)
    selected_data['Cluster'] = kmeans_selected.fit_predict(selected_data)

    # Plot the centroids and ellipses for each cluster with the specified color
    for cluster_id in range(num_clusters):
        cluster_data = selected_data[selected_data['Cluster'] == cluster_id]

        # Calculate centroid
        centroid_x = cluster_data['PC1'].mean()
        centroid_y = cluster_data['PC3'].mean()

        # Plot centroid
        plt.scatter(centroid_x, centroid_y, marker='X', s=200, color=centroid_colors.get(selected_columns[0], 'gray'), edgecolor='k', linewidth=1)

        # Calculate covariance matrix
        cluster_points = cluster_data[['PC1', 'PC3']].values
        cov_matrix = np.cov(cluster_points.T)

        # Calculate ellipse parameters
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigenvalues)

        # Plot the ellipse
        ellipse = Ellipse((centroid_x, centroid_y), width, height, angle=angle, edgecolor=outline_color, facecolor='none', linewidth=2)
        plt.gca().add_patch(ellipse)

# Set labels with PC1 and PC3 values
plt.xlabel(f'Principal Component 1 ({pca_3.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'Principal Component 3 ({pca_3.explained_variance_ratio_[2]*100:.2f}%)')

# Title
plt.title('Principal Component Analysis', fontweight='bold')

# Add grids
plt.grid(True)

# Display the legend
legend_labels = ['Baseline', 'DrugA']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label) for color, label in zip(['MediumVioletRed', 'lightpink', 'DimGray'], legend_labels)]
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handles=legend_handles, title='Legend')

plt.tight_layout()
plt.savefig('CSF_sigs_BL_DrugA_centroids_PCA_PC1_PC3.png')

# Show the plot
plt.show()

In [ ]:
Delta_Serum_data=pd.read_csv("delta serum.csv", index_col=0)
Delta_Serum_data.head(7000000)

In [ ]:
Delta_Serum_data1=Delta_Serum_data.T
Delta_Serum_data1.to_csv("Delta_Serum_data1.csv", index=True)
Delta_Serum_data1

In [ ]:
Score_Serum_data=pd.read_csv("Delta_Serum_data2.csv", index_col=0)
Score_Serum_data.head(7000000)

In [ ]:
Protein1 = Score_Serum_data[['ACHE','prog rate']]
Protein1

In [ ]:
clustermap = sns.clustermap(Protein1, annot=True, fmt=".4f", cmap="Blues", xticklabels=True, yticklabels=True)

In [ ]:
correlation_matrix5=Protein1.corr()
correlation_matrix5

In [ ]:
correlation_matrix4 = Score_Serum_data.corr()
correlation_matrix4

In [ ]:
# Plot the clustermap with consistent x and y tick labels
clustermap = sns.clustermap(correlation_matrix4, annot=True, fmt=".4f", cmap="Blues", xticklabels=True, yticklabels=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Increase the size of the plot
plt.figure(figsize=(5, 25))  # Adjust the width and height as needed

# Your heatmap code
Prog_heatmap = sns.heatmap(correlation_matrix4[['prog rate']], annot=False, fmt=".4f", cmap="RdBu", xticklabels=True, yticklabels=True)

# Save the figure
plt.savefig('heatmap.png')

plt.show()


In [ ]:
Delta_Serum_sigs=pd.read_csv("Delta_Serum_sigs.csv", index_col=0)
Delta_Serum_sigs.head(7000000)

In [ ]:
correlation_matrix5 = Delta_Serum_sigs.corr()
correlation_matrix5

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Increase the size of the plot
plt.figure(figsize=(10, 8))  # Adjust the width and height as needed

# Your heatmap code
Prog_heatmap = sns.heatmap(correlation_matrix5[['prog rate']], annot=False, fmt=".4f", cmap="RdBu", xticklabels=True, yticklabels=True)

plt.show()


In [ ]:
Marker_Score_ratio_data=pd.read_csv("Marker_Score_ratio.csv", index_col=0)
Marker_Score_ratio_data.head(7000000)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define a custom colormap from blue to white to red
cmap_colors = [(0, 'blue'), (0.5, 'white'), (1, 'red')]
custom_cmap = LinearSegmentedColormap.from_list("custom_colormap", cmap_colors)

# Assuming Serum_CSF_common_proteins is your data
Marker_Score_ratio_data_clustermap = sns.clustermap(Marker_Score_ratio_data, annot=False, fmt=".4f", cmap=custom_cmap, xticklabels=True, yticklabels=True,figsize=(12, 20))

plt.show()


In [ ]:
Marker_Prog_Score_ratio_data=pd.read_csv("Prog_Score_Serum.csv", index_col=0)
Marker_Prog_Score_ratio_data.head(7000000)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define a custom colormap from blue to white to red
cmap_colors = [(0, 'blue'), (0.5, 'white'), (1, 'red')]
custom_cmap = LinearSegmentedColormap.from_list("custom_colormap", cmap_colors)

# Assuming Serum_CSF_common_proteins is your data
Marker_Prog_Score_ratio_data_clustermap = sns.clustermap(Marker_Prog_Score_ratio_data, annot=False, fmt=".4f", cmap=custom_cmap, xticklabels=True, yticklabels=True,figsize=(12, 20))

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define a custom colormap from blue to white to red
cmap_colors = [(0, 'blue'), (0.5, 'white'), (1, 'red')]
custom_cmap = LinearSegmentedColormap.from_list("custom_colormap", cmap_colors)

# Assuming Serum_CSF_common_proteins is your data
Marker_Prog_Score_CSF_ratio_data_clustermap = sns.clustermap(Marker_Prog_Score_CSF_ratio_data, annot=False, fmt=".4f", cmap=custom_cmap, xticklabels=True, yticklabels=True,figsize=(12, 20))

plt.show()

In [ ]:
Marker_Prog_Score_prot_ratio_data=pd.read_csv("Prog_CSF_Protein.csv", index_col=0)
Marker_Prog_Score_prot_ratio_data.head(7000000)

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define a custom colormap from blue to white to red
cmap_colors = [(0, 'blue'), (0.5, 'white'), (1, 'red')]
custom_cmap = LinearSegmentedColormap.from_list("custom_colormap", cmap_colors)

# Assuming Serum_CSF_common_proteins is your data
Marker_Prog_Score_prot_ratio_data_clustermap = sns.clustermap(Marker_Prog_Score_prot_ratio_data, annot=False, fmt=".4f", cmap=custom_cmap, xticklabels=True, yticklabels=True,figsize=(12, 20))

plt.show()